# 1° Etapa do Projeto Prático

**Grupo:** Zero<br>
**Projeto:** Análise de Suporte em Comunidades de Saúde Mental no Reddit Brasileiro (Análise de Sentimentos)

Renato Calacina Spessotto - 14605824<br>
Edvandro Rodrigues Pereira - 18046338<br>
Guilherme Pascoale Godoy - 14576277<br>
Henrique Drago - 14675441<br>
Karl Cruz Altenhofen - 14585976<br>
Maicon Chaves Marques - 14593530<br>

**Pipeline:**

Normalização -> Tokenização

**->**

POS Tagging, Parsing de Dependência, Léxico -> Regras Linguísticas

**->**

Score -> Classificação

## Etapa 0 - Banco de Dados

In [ ]:
# Etapa 0 - Banco de Dados
import gdown
import json

Utilização de dois subreddits com comentários em PT-BR sobre saúde mental.

[r/AnsiedadeDepressao](https://www.reddit.com/r/AnsiedadeDepressao/) - 3.203 membros

Coleta realizada através do [PRAW: The Python Reddit API Wrapper](https://praw.readthedocs.io/en/stable/).

[r/desabafos](https://www.reddit.com/r/desabafos/) - 561.410 membros

Coleta realizada através dos [dumps](https://academictorrents.com/details/3e3f64dee22dc304cdd2546254ca1f8e8ae542b4) do [Pushshift Reddit](https://github.com/pushshift/api). Base disponível para [subreddits](https://www.reddit.com/r/pushshift/comments/1r5z42j/comment/o5mjcvn/) considerados populares.

Os 04 códigos abaixo foram utilizados para a coleta dos dados. Contudo, as credências da API não serão disponibilizadas. Da mesma forma dos dumps não serão trazidos aqui devido ao seu tamanho (>10GB). Portanto os códigos aparecem de forma comentada.

In [ ]:
# @title
"""#Posts de AnsiedadeDepressao
import praw
import json
import time

reddit = praw.Reddit()

subreddit = reddit.subreddit("AnsiedadeDepressao")

resultado = []

for submission in subreddit.top(time_filter="all", limit=None):

    # filtro: mais de 10 comentários
    if submission.num_comments <= 10:
        continue

    post_dict = {
        "id": submission.id,
        "subreddit": str(submission.subreddit),
        "subreddit_id": submission.subreddit_id,
        "author": str(submission.author) if submission.author else None,

        "title": submission.title,
        "selftext": submission.selftext,

        "score": submission.score,
        "created_utc": submission.created_utc,
        "retrieved_on": int(time.time()),

        "permalink": f"https://www.reddit.com{submission.permalink}",
        "url": submission.url,
        "domain": submission.domain,

        "is_self": submission.is_self,
        "over_18": submission.over_18,
        "num_comments": submission.num_comments,

        "ups": submission.ups,
        "downs": submission.downs,
        "gilded": submission.gilded,

        "stickied": submission.stickied,
        "edited": submission.edited,

        "author_flair_text": submission.author_flair_text,
        "author_flair_css_class": submission.author_flair_css_class,
        "link_flair_text": submission.link_flair_text,
        "link_flair_css_class": submission.link_flair_css_class
    }

    resultado.append(post_dict)

print(f"Total coletado: {len(resultado)} posts")

# salva
with open("output.json", "w", encoding="utf-8") as f:
    json.dump(resultado, f, ensure_ascii=False, indent=4)"""

In [ ]:
# @title
"""#Comentários de AnsiedadeDepressao
import praw
import json
import time

reddit = praw.Reddit()

# 🔹 lista de IDs dos posts
post_ids = ['1rcidns', '141jogu', '13dm2ks', '13g2gpf', '1huoy84', '1lve0g8', '1oy0nhw', '1ogxeex', '1m8hoin', '1373164', '1qop1sl', '13cva5g', '1rl08l5', '1pvvwfe', '1k74w4r', '1lvmmiz', '1k9k9uy', '1sbozy2', '1oh3yww', '1ls9th9', '1rcy7jq', '1r9uepy', '13dabxr', '1363gkm', '1qhrjt4', '1qazdn2', '1m9zico', '1r98h44', '1n2sgvw', '1mit6w2', '1m8ggb6', '1m5mpej', '1ll4z33', '1ka6hon', 'rfckxd', '1qlbw0g', '1ptd47r', 'vqnnlo', 'pxmmx9', '1r06fzl', '1qqapgs', '1qa01eh', '1nf7sqw', '1ne7hqg', '1ndu827', '1m0ye4h', '1lz254j', '1k3rsrz', '1jp7q6j']

resultado = []

for post_id in post_ids:
    submission = reddit.submission(id=post_id)

    submission.comments.replace_more(limit=None)

    for comment in submission.comments.list():

        # ignora objetos inválidos
        if not isinstance(comment, praw.models.Comment):
            continue

        comment_dict = {
            "id": comment.id,
            "name": comment.name,
            "author": str(comment.author) if comment.author else None,
            "body": comment.body,

            "subreddit": str(comment.subreddit),
            "subreddit_id": comment.subreddit_id,

            "link_id": comment.link_id,
            "parent_id": comment.parent_id,

            "created_utc": comment.created_utc,
            "retrieved_on": int(time.time()),

            "score": comment.score,
            "ups": comment.ups,
            "downs": comment.downs,

            "controversiality": comment.controversiality,
            "gilded": comment.gilded,

            "edited": comment.edited,
            "archived": comment.archived,
            "score_hidden": comment.score_hidden,

            "permalink": f"https://www.reddit.com{comment.permalink}"
        }

        resultado.append(comment_dict)

    print(f"Post {post_id} processado - {submission.num_comments} comentários")

print(f"\nTotal de comentários coletados: {len(resultado)}")

# salva
with open("output.json", "w", encoding="utf-8") as f:
    json.dump(resultado, f, ensure_ascii=False, indent=4)"""

In [ ]:
# @title
"""#Posts de desabafos
import json
import time
import re

arquivo_entrada = "desabafos_submissions"
arquivo_saida = "output.json"

resultado = []

# palavras proibidas (+18)
excluir = []

def tokenize(texto):
    if not texto:
        return set()
    return set(re.findall(r'\b\w+\b', texto.lower()))

palavras_set = set(p.lower() for p in palavras)
excluir_set = set(p.lower() for p in excluir)

with open(arquivo_entrada, "r", encoding="utf-8") as f:
    for linha in f:
        linha = linha.strip()
        if not linha:
            continue

        try:
            submission = json.loads(linha)
        except json.JSONDecodeError:
            continue

        # filtro 1
        if submission.get("num_comments", 0) <= 52:
            continue

        # filtro 2
        flair = submission.get("link_flair_text")
        if flair != "Depressão":
            continue

        titulo = submission.get("title", "")
        corpo = submission.get("selftext", "")

        tokens = tokenize(titulo) | tokenize(corpo)

        # filtro 3: NÃO pode conter palavras proibidas
        if not excluir_set.isdisjoint(tokens):
            continue

        post_dict = {
            "id": submission.get("id"),
            "subreddit": submission.get("subreddit"),
            "subreddit_id": submission.get("subreddit_id"),
            "author": submission.get("author"),

            "title": submission.get("title"),
            "selftext": submission.get("selftext"),

            "score": submission.get("score"),
            "created_utc": submission.get("created_utc"),
            "retrieved_on": submission.get("retrieved_on", int(time.time())),

            "permalink": f"https://www.reddit.com{submission.get('permalink')}" if submission.get("permalink") else None,
            "url": submission.get("url"),
            "domain": submission.get("domain"),

            "is_self": submission.get("is_self"),
            "over_18": submission.get("over_18"),
            "num_comments": submission.get("num_comments"),

            "ups": submission.get("ups"),
            "downs": submission.get("downs"),
            "gilded": submission.get("gilded"),

            "stickied": submission.get("stickied"),
            "edited": submission.get("edited"),

            "author_flair_text": submission.get("author_flair_text"),
            "author_flair_css_class": submission.get("author_flair_css_class"),
            "link_flair_text": flair,
            "link_flair_css_class": submission.get("link_flair_css_class"),

            "upvote_ratio": submission.get("upvote_ratio"),
            "distinguished": submission.get("distinguished"),
            "locked": submission.get("locked"),
            "spoiler": submission.get("spoiler")
        }

        resultado.append(post_dict)

print(f"Total coletado: {len(resultado)} posts")

with open(arquivo_saida, "w", encoding="utf-8") as f:
    json.dump(resultado, f, ensure_ascii=False, indent=4)"""

In [ ]:
# @title
"""#Comentários de desabafos
import json
import time

arquivo = "desabafos_comments"  #arquivo de entrada

# 🔹 IDs dos posts que você quer (sem o "t3_")
post_ids = ['h8xv06', 'hfcz4y', 'hsln5u', 'il4wuz', 'ivhl6b', 'ixl72z', 'katk91', 'ln1dmq', 'lsjssz', 'm01qc4', 'm5211p', 'm7bbth', 'ma44br', 'mwpi3p', 'ndzdws', 'nlpsm0', 'nwgqcv', 'ods4yq', 'og176d', 'ooctte', 'ove5eq', 'ovf3bn', 'owu963', 'p55e08', 'p5fety', 'p6nsfo', 'pf3lui', 'pg7o23', 'prcvhv', 'q273f5', 'q4n48v', 'q7fkp4', 'q7hp62', 'q7iovo', 'qd46qa', 'qhpeam', 'qpngb2', 'qqtrbe', 'qvwrkh', 'qysz90', 'r3pmja', 'rcj498', 'rmbzgq', 'rmvihq', 'rrgnm1', 'rt1yax', 'rwtma3', 'rxi3tm', 's1d5ar', 's9qfhl', 'sat696', 'se5g2l', 'sh5yfo', 'siaf96', 'swbw0x', 't1iwtd', 't4n805', 't7alxq', 'tc11od', 'tc5ao8', 'tgytnn', 'thcdsq', 'tkhu99', 'tmrdat', 'tosj4p', 'tpvk49', 'ttxh6i', 'tx4hxk', 'u2eb5z', 'u7hrg9', 'u7ze44', 'ua8jht', 'uyil3c', 'uznmxn', 'vfexht', 'vu8arj', 'w00gkr', 'w24918', 'w2plr7', 'w8pq84', 'w9z0vu', 'wf2s9z', 'wf8j7g', 'wmsoeu', 'wnjfui', 'wojg4k', 'wsjfmi', 'x3a1ft', 'xf4die', 'xgc775', 'xhnrih', 'xi68fi', 'xj6os6', 'xrm1nm', 'y314ez', 'y4u4ir', 'y5zf2w', 'y61fb3', 'ynt7i9', 'yo7g9b', 'yqxoo3', 'yrx6gp', 'yvdjou', 'yw7yqp', 'z7eeun', 'z9mlda', 'zb08vr', 'zcgsli', 'zdakob', 'zei6c8', 'zfam8w', 'zg0edz', 'zj3il7', 'zjxd5m', 'zkn7v9', 'zlyp1r', 'zm279d', 'znwlwy', 'zrnq8j', 'zvyo4q', '100bp6p', '100t8gi', '101tb6i', '103cfc5', '103ryo0', '104ih8c', '105encl', '105pmaa', '107r86f', '108m4hr', '1097aa3', '109bkdh', '10cv2v9', '10dh1pi', '10dowmx', '10dzv86', '10g5ar4', '10ins1v', '10ix53p', '10izll3', '10mz7ro', '10n5ezm', '10qiapb', '10sj6zm', '10t0hw4', '10vbuxi', '10vkwib', '10xbhct', '10yrvy2', '110nkdq', '112bm70', '113o7tm', '116twmn', '117xr5a', '11a5570', '11avlls', '11c4m1t', '11cepjy', '11di6ee', '11lw7pc', '11nbkkt', '11tx3en', '1205coj', '122lk92', '125jmmq', '127mlkt', '12hys26', '12t3ed9', '132hb9f', '13bf0uc', '13jlj9q', '14d1saq', '14dlq6j', '14kixul', '14m322b', '14r1st9', '158fxal', '1685ttn', '16huetm', '16jxzlk', '16n6a92', '16oii1a', '16tuguf', '170rpu0', '172p8rx', '17cow4s', '17e56bx', '17k4yp2', '17m342g', '17mu95x', '181ddts', '183m42h', '18b0cbl', '18urr3n', '1aghgl9', '1aj6rej', '1api59p', '1at69cd', '1ay390j', '1azz66k', '1b3dhhy', '1b7yrhj', '1bchwn4', '1bk46on', '1bklqmz', '1bxoc55', '1bxpnmy', '1c4l4l4', '1c9lsj3', '1cf8dmz', '1cfj1io', '1d70ifn', '1e2r9c1', '1e4aab0', '1e55xpk', '1e7vbxp', '1ecc64p', '1edqjrc', '1ee3nvy', '1eij6nu', '1es001i', '1evc4h5', '1fo25zj', '1fq6l55', '1fr4w4v', '1frc52v', '1fyhgi5', '1fz2dxn', '1fz4mhs', '1g3yjjx', '1g5f2ul', '1g6m2ul', '1ga4c1a', '1gdxhka', '1gkvmi4', '1gszx4d', '1h2k2uv', '1h3063x', '1hbi7fk', '1hne4r0', '1hqe6bl', '1hsulah', '1hxl1nd', '1i2a7bp', '1i3zogb', '1irh0rj', '1irlxbt', '1j4o1km', '1k6btya', '1kh6ei2', '1khs354', '1lqnmv4', '1osz8ee', '1owi1e2', '1oxzgy4', '1oze5sy', '1p6sn8h', '1pg6h27']

resultado = []

with open(arquivo, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        try:
            comment = json.loads(line)

            # 🔹 extrai post_id do link_id
            link_id = comment.get("link_id", "")
            post_id = link_id.replace("t3_", "")

            # 🔹 filtra pelos posts desejados
            if post_ids and post_id not in post_ids:
                continue

            # 🔥 reconstrói o permalink
            permalink = f"https://www.reddit.com/r/{comment.get('subreddit')}/comments/{post_id}/_/{comment.get('id')}"

            comment_dict = {
                "id": comment.get("id"),
                "name": comment.get("name"),
                "author": comment.get("author"),
                "body": comment.get("body"),

                "subreddit": comment.get("subreddit"),
                "subreddit_id": comment.get("subreddit_id"),

                "link_id": comment.get("link_id"),
                "parent_id": comment.get("parent_id"),

                "created_utc": int(comment.get("created_utc", 0)),
                "retrieved_on": comment.get("retrieved_on", int(time.time())),

                "score": comment.get("score"),
                "ups": comment.get("ups"),
                "downs": comment.get("downs"),

                "controversiality": comment.get("controversiality"),
                "gilded": comment.get("gilded"),

                "edited": comment.get("edited"),
                "archived": comment.get("archived"),
                "score_hidden": comment.get("score_hidden"),

                "permalink": permalink
            }

            resultado.append(comment_dict)

            # 🔹 log de progresso
            if i % 100000 == 0:
                print(f"{i} linhas processadas...")

        except json.JSONDecodeError:
            continue

print(f"\nTotal de comentários coletados: {len(resultado)}")

# 🔹 salva saída
with open("output.json", "w", encoding="utf-8") as f:
    json.dump(resultado, f, ensure_ascii=False, indent=4)"""

In [ ]:
file_id = '1t7gDsh20kp-5E2cxAG0X6Dcs2_PVWIRh'
gdown.download(f'https://drive.google.com/uc?id={file_id}', 'posts_AnsiedadeDepressao.json', quiet=False)

file_id = '1LZ0MbbZR2vDuCx-25fHHx13MgClLH-vV'
gdown.download(f'https://drive.google.com/uc?id={file_id}', 'comentarios_AnsiedadeDepressao.json', quiet=False)

file_id = '1Tdyp7LMwH9xluteGrThz-pTf8f9eBI7b'
gdown.download(f'https://drive.google.com/uc?id={file_id}', 'posts_desabafos.json', quiet=False)

file_id = '179Kf10ELpVwU8sL2ZIBu0-WtvREG7W7Q'
gdown.download(f'https://drive.google.com/uc?id={file_id}', 'comentarios_desabafos.json', quiet=False)

file_id = '1JTA1MGv_lihekH2nxWlNpTaBnQrEeaPM'
gdown.download(f'https://drive.google.com/uc?id={file_id}', 'abreviacoes.json', quiet=False)

file_id = '1dcfSSaoV5cT55KVH-CaOfdMEV_2NALHm'
gdown.download(f'https://drive.google.com/uc?id={file_id}', 'emojis_ptbr.json', quiet=False)

file_id = '1PqFwKKGag9OWLnwFX1nQ_DceCPWhbSjr'
gdown.download(f'https://drive.google.com/uc?id={file_id}', 'formas.saocarlos.txt', quiet=False)

file_id = '1ldHzSUU2Am-dLHRzqW6XBOIS7bGWzPt7'
gdown.download(f'https://drive.google.com/uc?id={file_id}', 'SentiLex-lem-PT02.txt', quiet=False)

file_id = '133cz6GbogY2vtF8efUQqAX_RXRIKbNVb'
gdown.download(f'https://drive.google.com/uc?id={file_id}', 'lexico_v3.0.txt', quiet=False)

### Filtragem e Limpeza dos dados

A filtragem inicial tem por objetivo:

- Remover comentários do próprio autor da publicação
- Remover comentários deletados
- Filtrar os comentários mantendo o limite de aproximadamente 10 mil

In [ ]:
#AnsiedadeDepressao

# arquivos
ARQ_POSTS = "posts_AnsiedadeDepressao.json"
ARQ_COMENTARIOS = "comentarios_AnsiedadeDepressao.json"
ARQ_SAIDA = "comentarios_AnsiedadeDepressao_filtrados.json"

LIMITE = 10000  # limite de comentários

# -----------------------------
# 1. Carregar posts
# -----------------------------
with open(ARQ_POSTS, "r", encoding="utf-8") as f:
    posts = json.load(f)

post_author = {p["id"]: p["author"] for p in posts}

# -----------------------------
# 2. Carregar comentários
# -----------------------------
with open(ARQ_COMENTARIOS, "r", encoding="utf-8") as f:
    comentarios = json.load(f)

comentarios_filtrados = []

# -----------------------------
# 3. Filtrar
# -----------------------------
for c in comentarios:
    if len(comentarios_filtrados) >= LIMITE:
        break

    autor_comentario = c.get("author")
    texto = c.get("body", "").lower()  # 🔹 normaliza

    link_id = c.get("link_id", "")
    post_id = link_id.replace("t3_", "")

    autor_post = post_author.get(post_id)

    # regras
    if autor_comentario is None:
        continue

    if autor_comentario == "[deleted]":
        continue

    if autor_post and autor_comentario == autor_post:
        continue

    comentarios_filtrados.append(c)

# -----------------------------
# 4. Salvar
# -----------------------------
with open(ARQ_SAIDA, "w", encoding="utf-8") as f:
    json.dump(comentarios_filtrados, f, ensure_ascii=False, indent=4)

print(f"Total original: {len(comentarios)}")
print(f"Total filtrado: {len(comentarios_filtrados)}")

In [ ]:
#desabafos

# arquivos
ARQ_POSTS = "posts_desabafos.json"
ARQ_COMENTARIOS = "comentarios_desabafos.json"
ARQ_SAIDA = "comentarios_desabafos_filtrados.json"

LIMITE = 99999  # limite de comentários

# -----------------------------
# 1. Carregar posts
# -----------------------------
with open(ARQ_POSTS, "r", encoding="utf-8") as f:
    posts = json.load(f)

post_author = {p["id"]: p["author"] for p in posts}

# -----------------------------
# 2. Carregar comentários
# -----------------------------
with open(ARQ_COMENTARIOS, "r", encoding="utf-8") as f:
    comentarios = json.load(f)

comentarios_filtrados = []

# controle
limite_atingido = False
post_limite = None

# -----------------------------
# 3. Filtrar
# -----------------------------
for c in comentarios:

    link_id = c.get("link_id", "")
    post_id = link_id.replace("t3_", "")

    # se já passou do limite, só continua até trocar de post
    if limite_atingido:
        if post_id != post_limite:
            break  # saiu do post → pode parar
    else:
        if len(comentarios_filtrados) >= LIMITE:
            limite_atingido = True
            post_limite = post_id  # guarda o post atual

    autor_comentario = c.get("author")
    texto = c.get("body", "").lower()

    autor_post = post_author.get(post_id)

    # regras
    if autor_comentario is None:
        continue

    if autor_comentario == "[deleted]":
        continue

    if autor_post and autor_comentario == autor_post:
        continue

    comentarios_filtrados.append(c)

# -----------------------------
# 4. Salvar
# -----------------------------
with open(ARQ_SAIDA, "w", encoding="utf-8") as f:
    json.dump(comentarios_filtrados, f, ensure_ascii=False, indent=4)

print(f"Total original: {len(comentarios)}")
print(f"Total filtrado: {len(comentarios_filtrados)}")

In [ ]:
with open("comentarios_AnsiedadeDepressao_filtrados.json", "r", encoding="utf-8") as f:
    a = json.load(f)

with open("comentarios_desabafos_filtrados.json", "r", encoding="utf-8") as f:
    b = json.load(f)

resultado = a + b
print(len(resultado))

with open("corpus.json", "w", encoding="utf-8") as f:
    json.dump(resultado, f, ensure_ascii=False, indent=4)

## Etapa 01 — Normalização
- Converte texto para minúsculas (`aplicar_lower`)
- Remove acentos (ex: "ação" → "acao", "é" → "e") (`aplicar_remover_acentos`)
- Substitui emojis (Unicode e textuais) por palavras (`aplicar_emojis`)
- Opcionalmente remove emojis (`remover_emojis`)
- Normaliza pontuação (remove repetições como "!!!" → "!") (`aplicar_pontuacao`)
- Remove pontuação irrelevante (mantém ! ? .) (`aplicar_pontuacao`)
- Tokeniza o texto usando NLTK (`aplicar_tokenizacao`)
- Expande abreviações (ex: "vc" → "você") (`aplicar_abreviacoes`)
- Converte números para texto (ex: "10" → "dez") (`aplicar_numeros`)
- Corrige palavras com SymSpell (`aplicar_correcao`)
- Aplica lematização com spaCy (ex: "correndo" → "correr") (`aplicar_lematizacao`)
- Reconstrói o texto a partir dos tokens

In [ ]:
# Instala dependências necessárias para a normalização do texto
# - symspellpy: correção ortográfica
# - num2words: conversão de números para palavras

!pip install symspellpy num2words
!pip install -q spacy symspellpy num2words nltk
!python -m spacy download pt_core_news_sm

In [ ]:
import re
import unicodedata
from pathlib import Path

import nltk
from nltk.tokenize import word_tokenize

import spacy

from symspellpy import SymSpell, Verbosity
from num2words import num2words

nltk.download("punkt")
nltk.download("punkt_tab")

In [ ]:
# =========================
# spaCy Lematizador
# =========================
nlp = spacy.load("pt_core_news_sm")

# =========================
# SymSpell
# =========================
sym_spell = SymSpell(max_dictionary_edit_distance=2, prefix_length=7)
sym_spell.load_dictionary("formas.saocarlos.txt", 0, 1)

def corrigir_token(token: str) -> str:
    if len(token) <= 2:
        return token

    sugestoes = sym_spell.lookup(token, Verbosity.TOP, max_edit_distance=2)

    if not sugestoes:
        return token

    melhor = sugestoes[0]

    if melhor.distance == 1 and melhor.count > 1000:
        return melhor.term

    return token


# =========================
# Regras adicionais
# =========================

# reduzir letras repetidas >3 para 1
def reduzir_repeticoes(texto: str) -> str:
    return re.sub(r'(.)\1{2,}', r'\1', texto)


# remover caracteres especiais dentro de palavras
def remover_especiais_dentro_palavra(texto: str) -> str:
    return re.sub(r'(?<=\w)[^\w\s]+(?=\w)', '', texto)


# unir palavras separadas por caracteres especiais
def unir_palavras_separadas(texto: str) -> str:
    return re.sub(
        r'\b(?:\w[\-_\.\*]){1,}\w\b',
        lambda m: re.sub(r'[\-_\.\*]', '', m.group()),
        texto
    )


# =========================
# Remover Acentos
# =========================
def remover_acentos(texto: str) -> str:
    return ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )


# =========================
# Loaders
# =========================
def carregar_json(caminho):
    with open(caminho, "r", encoding="utf-8") as f:
        return json.load(f)


abreviacoes = {k.lower(): v.lower() for k, v in carregar_json("abreviacoes.json").items()}
emojis = carregar_json("emojis_ptbr.json")


# =========================
# Normalização
# =========================
def normalizar_texto(
    texto: str,
    aplicar_lower=True,
    aplicar_remover_acentos=False,
    aplicar_emojis=True,
    remover_emojis=False,
    aplicar_pontuacao=True,
    aplicar_tokenizacao=True,
    aplicar_abreviacoes=True,
    aplicar_numeros=True,
    aplicar_correcao=True,
    aplicar_lematizacao=False
) -> str:

    # Lowercase
    if aplicar_lower:
        texto = texto.lower()

    # Remover acentos
    if aplicar_remover_acentos:
        texto = remover_acentos(texto)

    # Emojis
    if aplicar_emojis:
        for emoji, descricao in sorted(emojis.items(), key=lambda x: -len(x[0])):
            if emoji in texto:
                if remover_emojis:
                    texto = texto.replace(emoji, " ")
                else:
                    texto = texto.replace(emoji, f" ({descricao}) ")

    # reduzir repetição de letras
    texto = reduzir_repeticoes(texto)

    # remover especiais dentro da palavra
    texto = remover_especiais_dentro_palavra(texto)

    # unir palavras separadas por símbolos
    texto = unir_palavras_separadas(texto)

    # =========================

    # Pontuação
    if aplicar_pontuacao:
        texto = re.sub(r'([!?\.])\1+', r'\1', texto)
        texto = re.sub(r'[^\w\s!?\.]', ' ', texto)

    # Tokenização
    if aplicar_tokenizacao:
        tokens = word_tokenize(texto, language="portuguese")
    else:
        tokens = texto.split()

    # Abreviações
    if aplicar_abreviacoes:
        tokens = [
            palavra
            for token in tokens
            for palavra in abreviacoes.get(token, token).split()
        ]

    # Números
    if aplicar_numeros:
        resultado = []
        for t in tokens:
            if t.isdigit():
                try:
                    resultado.extend(num2words(int(t), lang="pt_BR").split())
                    continue
                except:
                    pass
            resultado.append(t)
    else:
        resultado = tokens

    # Correção
    if aplicar_correcao:
        tokens = [corrigir_token(t) for t in resultado]
    else:
        tokens = resultado

    # Lematização com spaCy
    if aplicar_lematizacao:
        doc = nlp(" ".join(tokens))
        tokens = [token.lemma_ for token in doc]

    return " ".join(tokens)


# =========================
# Processamento de corpus
# =========================
def processar_corpus(
    caminho_corpus="corpus.json",
    limite=None,
    **kwargs
):
    corpus = carregar_json(caminho_corpus)

    if limite is not None:
        corpus = corpus[:limite]

    resultados = []

    for item in corpus:
        original = item.get("body", "")
        normalizado = normalizar_texto(original, **kwargs)

        resultados.append({
            "original": original,
            "normalizado": normalizado
        })

    return resultados


# =========================
# Execução
# =========================
resultados = processar_corpus(
    limite=29999,
    aplicar_lower=True,
    aplicar_remover_acentos=True,
    aplicar_emojis=True,
    remover_emojis=False,
    aplicar_pontuacao=False,
    aplicar_tokenizacao=True,
    aplicar_abreviacoes=True,
    aplicar_numeros=True,
    aplicar_correcao=True,
    aplicar_lematizacao=False
)

# Exemplo de consumo
print(f"Total: {len(resultados)}\n")
for r in resultados[:50]:
    print("Original:", r["original"])
    print("Normalizado:", r["normalizado"])
    print("-" * 50)


## Etapa 02 — Análise Linguística (POS Tagging, Parsing e Léxico)                                                                         


In [ ]:
# =========================
# Instalação para POS Tagging e Léxico
# =========================

!pip install -q pandas

import pandas as pd

### POS-Tagging

In [ ]:
# =========================
# POS TAGGING COMPLETO
# =========================

resultado_pos = []

for item in resultados:
    doc = nlp(item["normalizado"])

    tokens = []
    for token in doc:
        if token.is_space:
            continue

        tokens.append({
            "token": token.text,
            "lemma": token.lemma_,
            "pos": token.pos_,
            "tag": token.tag_
        })

    resultado_pos.append({
        "original": item["original"],
        "normalizado": item["normalizado"],
        "tokens_pos": tokens
    })

# salvar arquivo
with open("corpus_pos_tagging.json", "w", encoding="utf-8") as f:
    json.dump(resultado_pos, f, ensure_ascii=False, indent=4)

# mostrar exemplo
print("POS Tagging concluído")
print(json.dumps(resultado_pos[0], ensure_ascii=False, indent=4)[:2000])

### Parsing de Dependencias

In [ ]:
# =========================
# PARSING DE DEPENDENCIAS NO CORPUS
# =========================

def carregar_modelo_pt():
    for nome in ('pt_core_news_lg', 'pt_core_news_sm'):
        try:
            modelo = spacy.load(nome)
            if 'parser' in modelo.pipe_names:
                return modelo
        except OSError:
            pass
    raise RuntimeError("Nenhum modelo PT do spaCy com parser encontrado. Instale com: python -m spacy download pt_core_news_sm")


if 'nlp' not in globals() or 'parser' not in nlp.pipe_names:
    nlp = carregar_modelo_pt()


def relacao_simplificada(dep):
    if dep in {'neg', 'advmod', 'amod'}:
        return 'modifica'
    return 'relaciona'


def analisar_dependencias_texto(texto):
    doc = nlp(texto)
    relacoes = []

    for t in doc:
        if t.is_space:
            continue

        relacoes.append({
            'origem': t.text,
            'origem_i': t.i,
            'origem_lemma': t.lemma_,
            'origem_pos': t.pos_,
            'relacao': relacao_simplificada(t.dep_),
            'dep': t.dep_,
            'destino': t.head.text,
            'destino_i': t.head.i,
            'destino_lemma': t.head.lemma_,
            'destino_pos': t.head.pos_
        })

    return relacoes


# Aplica nas mesmas frases usadas ao longo do pipeline.
fonte = resultado_pos if 'resultado_pos' in globals() else resultados

resultado_dependencias = []
for item in fonte:
    texto = item.get('normalizado', '')
    relacoes = analisar_dependencias_texto(texto)

    saida = {
        'original': item.get('original', ''),
        'normalizado': texto,
        'dependencias': relacoes
    }

    if 'tokens_pos' in item:
        saida['tokens_pos'] = item['tokens_pos']

    resultado_dependencias.append(saida)


with open('corpus_dependencias.json', 'w', encoding='utf-8') as f:
    json.dump(resultado_dependencias, f, ensure_ascii=False, indent=4)

print('Parsing de dependencias aplicado ao corpus.')
print(f'Total de frases processadas: {len(resultado_dependencias)}')
if resultado_dependencias:
    print(json.dumps(resultado_dependencias[0], ensure_ascii=False, indent=4)[:3000])

# Siglas de relacao sintatica (dep_) mais comuns no parser:
# neg    = marcador de negacao (ex.: "nao")
# advmod = modificador adverbial
# amod   = modificador adjetival
# nsubj  = sujeito nominal
# obj    = objeto direto
# iobj   = objeto indireto
# obl    = complemento obliquo/circunstancial
# cop    = verbo de ligacao
# xcomp  = complemento aberto do verbo
# ccomp  = complemento oracional
# conj   = termo coordenado
# cc     = conjuncao coordenativa

### Construção do Lexico

In [ ]:
# Código para obter as classes do OpLexicon (apenas para a construção do código)

def obter_classes_unicas_oplexicon(caminho):
    classes_unicas = set() # Usamos um set para evitar duplicatas

    try:
        with open(caminho, "r", encoding="utf-8") as f:
            for linha in f:
                linha = linha.strip()

                # Pula linhas vazias
                if not linha:
                    continue

                partes = linha.split(",")

                # A classe está na segunda coluna (índice 1)
                if len(partes) >= 2:
                    classe = partes[1].strip().upper()
                    if classe:
                        classes_unicas.add(classe)

        return sorted(list(classes_unicas))

    except FileNotFoundError:
        print(f"Erro: O arquivo '{caminho}' não foi encontrado.")
        return []

# Execução
CAMINHO_OPLEXICON = "lexico_v3.0.txt"
classes = obter_classes_unicas_oplexicon(CAMINHO_OPLEXICON)

print("Classes encontradas no OpLexicon:")
print(classes)

In [ ]:
# OpLexicon
CAMINHO_OPLEXICON = "lexico_v3.0.txt"

def carregar_oplexicon(caminho):
    lexico = {}

    # Definimos as classes que queremos manter
    CLASSES_DESEJADAS = {"ADJ", "VB"}

    with open(caminho, "r", encoding="utf-8") as f:
        for linha in f:
            linha = linha.strip()

            # Pula linhas vazias
            if not linha:
                continue

            partes = [p.strip() for p in linha.split(",")]

            # Formato esperado: termo, classe, polaridade, fonte
            if len(partes) < 3:
                continue

            # Extração dos dados
            termo = partes[0].lower()
            classe = partes[1].upper()

            # Se a classe não for ADJ ou VB, ignoramos a linha
            if classe not in CLASSES_DESEJADAS:
                continue

            polaridade_str = partes[2]

            try:
                polaridade = int(polaridade_str)
            except ValueError:
                continue

            # Estrutura para suportar palavras com múltiplas entradas (ex: 'estar' como VB)
            if termo not in lexico:
                lexico[termo] = []

            lexico[termo].append({
                "classe": classe,
                "polaridade": polaridade
            })

    return lexico

# Execução
lex_oplexicon = carregar_oplexicon(CAMINHO_OPLEXICON)

print(f"Total de termos (ADJ/VB) carregados: {len(lex_oplexicon)}")

# Exemplo de visualização de um termo
exemplo = "estar"
if exemplo in lex_oplexicon:
    print(f"Exemplo '{exemplo}':", lex_oplexicon[exemplo])

In [ ]:
# Código para obter as classes do SentiLex (apenas para a construção do código)
def obter_classes_unicas_sentilex(caminho):
    classes_unicas = set()

    try:
        # É fundamental usar iso-8859-1 para evitar erros de decodificação
        with open(caminho, "r", encoding="iso-8859-1") as f:
            for linha in f:
                linha = linha.strip()

                # Verifica se a linha contém a tag de classe gramatical
                if not linha or ".PoS=" not in linha:
                    continue

                # No SentiLex, o lema é separado dos metadados por um ponto
                # Ex: abafado.PoS=Adj;...
                partes = linha.split(".", 1)
                if len(partes) < 2:
                    continue

                # As tags de metadados são separadas por ponto e vírgula
                tags = partes[1].split(";")
                for tag in tags:
                    if "PoS=" in tag:
                        # Extrai o valor após o '=' (ex: 'Adj', 'V', 'N')
                        classe = tag.split("=")[1].strip().upper()
                        if classe:
                            classes_unicas.add(classe)

        return sorted(list(classes_unicas))

    except FileNotFoundError:
        print(f"Erro: O arquivo '{caminho}' não foi encontrado.")
        return []

# Execução
CAMINHO_SENTILEX = "SentiLex-lem-PT02.txt"
classes_sentilex = obter_classes_unicas_sentilex(CAMINHO_SENTILEX)

print("Classes encontradas no SentiLex-PT:")
print(classes_sentilex)

In [ ]:
# SentiLex
def carregar_sentilex(caminho):
    lexico = {}

    with open(caminho, 'r', encoding='iso-8859-1') as f:
        for linha in f:
            linha = linha.strip()
            if not linha or ".PoS=" not in linha:
                continue

            # Separa Lema e Metadados
            partes = linha.split('.', 1)
            lema = partes[0].lower()
            metadata = partes[1].split(';')

            pos = None
            pol_n0 = None
            pol_n1 = None
            pol_geral = None

            for info in metadata:
                # Mapeamento de POS (Adj -> ADJ, V -> VB)
                if 'PoS=' in info:
                    val_pos = info.split('=')[1].lower()
                    if val_pos == 'adj': pos = 'ADJ'
                    elif val_pos == 'v': pos = 'VB'
                    elif val_pos == 'n': pos = 'N'

                # Captura de polaridades específicas
                try:
                    if 'POL:N0=' in info:
                        pol_n0 = int(info.split('=')[1])
                    elif 'POL:N1=' in info:
                        pol_n1 = int(info.split('=')[1])
                    elif 'POL=' in info:
                        pol_geral = int(info.split('=')[1])
                except:
                    continue

            # Lógica de Prioridade para Desabafos
            # 1. Prioridade ao N0 (Quem fala/age)
            # 2. Fallback para POL (Polaridade geral)
            # 3. Último caso N1 (O objeto da ação)
            if pol_n0 is not None:
                polaridade = pol_n0
            elif pol_geral is not None:
                polaridade = pol_geral
            else:
                polaridade = pol_n1

            if pos and polaridade is not None:
                if lema not in lexico:
                    lexico[lema] = []
                lexico[lema].append({"classe": pos, "polaridade": polaridade})

    return lexico

# Uso
lex_sentilex = carregar_sentilex("SentiLex-lem-PT02.txt")
print(f"Total de lemas carregados: {len(lex_sentilex)}")

## Atribuição do Score

In [ ]:
# Usado na construção do código
def extrair_tags_unicas(dados_dependencias):
    tags_dep = set()
    tags_pos = set()

    for item in dados_dependencias:
        relacoes = item.get("dependencias", [])

        for rel in relacoes:
            # Coleta a tag de dependência
            if "dep" in rel:
                tags_dep.add(rel["dep"])

            # Coleta as tags de POS (origem e destino)
            if "origem_pos" in rel:
                tags_pos.add(rel["origem_pos"])
            if "destino_pos" in rel:
                tags_pos.add(rel["destino_pos"])

    return sorted(list(tags_dep)), sorted(list(tags_pos))

# Execução
lista_dep, lista_pos = extrair_tags_unicas(resultado_dependencias)

print("Tags de Dependência únicas:")
print(lista_dep)

print("\nTags de POS únicas:")
print(lista_pos)

In [ ]:
# Usado na construção do código

# Nossas listas de interesse
negadores = ["não", "nem", "nunca", "jamais", "nada", "tampouco", "sem"]
intensificadores = ["muito", "demais", "extremamente", "bastante", "tão", "pouco", "quase", "ligeiramente", "meio"]
verbos_neutros = ["ser", "estar", "ter", "haver", "ficar", "ir", "vir", "tentar", "parecer", "voltar", "passar", "continuar"]

def mapear_tags_modificadores(corpus, conjunto_alvo):

    alvos = set(conjunto_alvo)

    # Dicionário para guardar as descobertas: {palavra: {(POS, DEP), (POS, DEP)}}
    mapeamento = {}

    for item in corpus:
        relacoes = item.get("dependencias", [])

        for rel in relacoes:
            # Verificamos a palavra de origem (quem está "pendurado" em alguém)
            termo = rel["origem"].lower()

            if termo in alvos:
                pos = rel["origem_pos"]
                dep = rel["dep"]

                if termo not in mapeamento:
                    mapeamento[termo] = set()

                # Adicionamos a combinação única
                mapeamento[termo].add((pos, dep))

    # Formatação para exibição
    print(f"{'PALAVRA':<15} | {'POS/DEP ENCONTRADOS'}")
    print("-" * 50)

    for termo in sorted(mapeamento.keys()):
        combos = [f"[{p}/{d}]" for p, d in mapeamento[termo]]
        print(f"{termo:<15} | {' '.join(combos)}")

# Execução
mapear_tags_modificadores(resultado_dependencias, negadores)
print()
mapear_tags_modificadores(resultado_dependencias, intensificadores)
print()
mapear_tags_modificadores(resultado_dependencias, verbos_neutros)

In [ ]:
# Calculo dos Sentimentos
# --- CONFIGURAÇÕES DE OPERADORES ---
MULTIPLICADOR_EXCLAMACAO = 1.2
NEGADORES = {"não", "nem", "nunca", "jamais", "sem"}
VERBOS_NEUTROS = {
    # Existenciais e Estado
    "ser", "estar", "ter", "haver", "ficar", "ir", "vir", "parecer", "voltar", "passar", "continuar",

    # Modais e Necessidade
    "querer", "poder", "conseguir", "dever", "precisar", "tentar",

    # Percepção e Cognição
    "ver", "achar", "pensar", "sentir", "saber", "conhecer",

    # Aspectuais e Funcionais
    "começar", "dar", "levar", "fazer", "esperar", "andar"
}
INTENSIFICADORES = {
    "muito": 2.0,
    "bastante": 2.0,
    "extremamente": 2.5,
    "tão": 1.5,
    "demais": 2.0,
    "realmente": 1.5,
    "totalmente": 2.0,
    "completamente": 2.0,
    "profundamente": 2.5,
    "absurdamente": 2.5
}
ATENUADORES = {
    "quase": 0.5,
    "ligeiramente": 0.5,
    "pouco": 0.5,
    "meio": 0.5,
    "apenas": 0.5,
    "só": 0.5,
    "meramente": 0.5,
    "minimamente": 0.5
}
MAPA_TAGS = {
    "ADJ": ["ADJ"],
    "VERB": ["VB"],
    "AUX": ["VB"],
    "NOUN": ["N"]
}
PESOS_CLASSE = {
    "ADJ": 1.0,
    "NOUN": 0.8,
    "VERB": 0.4,
    "AUX": 0.0
}

def buscar_polaridade(lema, pos_spacy, lexico):
    """Busca polaridade com filtros de blacklist e normalização."""
    # Se for um verbo neutro, score zero imediatamente
    if pos_spacy in ["VERB", "AUX"] and lema in VERBOS_NEUTROS:
        return 0

    if lema in lexico:
        classes_aceitas = MAPA_TAGS.get(pos_spacy, [])
        for entrada in lexico[lema]:
            if entrada['classe'] in classes_aceitas:
                return entrada['polaridade']
    return 0

def calcular_sentimento_sintatico(corpus, lexico):
    resultado_final = []

    for item in corpus:
        texto = item.get("original", "")
        deps = item.get("dependencias", [])
        tokens_pos = item.get("tokens_pos", [])

        score_total = 0.0
        termos_identificados = []

        # Processamos cada token que pode ter carga sentimental
        for idx, tk in enumerate(tokens_pos):
            lema = tk['lemma']
            pos = tk['pos']

            polaridade_base = buscar_polaridade(lema, pos, lexico)

            if polaridade_base != 0:
                peso = PESOS_CLASSE.get(pos, 1.0)
                valor_termo = polaridade_base * peso
                modificador = 1.0

                # A. Modificadores Filhos (advmod)
                for d in deps:
                    if d['destino_i'] == idx and d['dep'] == 'advmod':
                        origem_txt = d['origem'].lower()
                        if d['origem_pos'] == 'ADV':
                            if origem_txt in NEGADORES: modificador *= -1
                            elif origem_txt in INTENSIFICADORES: modificador *= INTENSIFICADORES[origem_txt]
                            elif origem_txt in ATENUADORES: modificador *= ATENUADORES[origem_txt]

                    # Inclui o sinal de exclamação "!" como um multiplicador
                    elif d['dep'] == 'punct' and '!' in d['origem']:
                            modificador *= MULTIPLICADOR_EXCLAMACAO

                # B. Herança de Negação (Pai para Adjetivo)
                if pos == "ADJ":
                    pai_id = next((d['destino_i'] for d in deps if d['origem_i'] == idx), None)
                    if pai_id is not None:
                        for d in deps:
                            if d['destino_i'] == pai_id and d['origem'].lower() in NEGADORES:
                                modificador *= -1
                                break

                valor_final = valor_termo * modificador
                score_total += valor_final
                termos_identificados.append(f"{lema}({round(valor_final, 2)})")

        # Classificação Final com limites ajustados
        if score_total > 0.4: label = "Positivo"
        elif score_total < -0.4: label = "Negativo"
        else: label = "Neutro"

        resultado_final.append({
            "texto": texto,
            "score": round(score_total, 2),
            "sentimento": label,
            "detalhes": termos_identificados
        })

    return resultado_final

In [ ]:
# OpLexicon
final_oplex = calcular_sentimento_sintatico(resultado_dependencias, lex_oplexicon)

found_oplex = {}
for r in final_oplex:
    classe = r["sentimento"]
    if classe not in found_oplex:
        found_oplex[classe] = 0
    found_oplex[classe] += 1

print(found_oplex)
print()

# Visualizando os primeiros 20 resultados
for r in final_oplex[:50]:
    print(f"[{r['sentimento']}] Score: {r['score']} | Texto: {r['texto']}")
    print(f"   -> Termos: {r['detalhes']}\n")

In [ ]:
# SentiLex
final_sentilex = calcular_sentimento_sintatico(resultado_dependencias, lex_sentilex)

found_sentilex = {}
for r in final_sentilex:
    classe = r["sentimento"]
    if classe not in found_sentilex:
        found_sentilex[classe] = 0
    found_sentilex[classe] += 1

print(found_sentilex)
print()

# Visualizando os primeiros 20 resultados
for r in final_sentilex[:50]:
    print(f"[{r['sentimento']}] Score: {r['score']} | Texto: {r['texto']}")
    print(f"   -> Termos: {r['detalhes']}\n")